# Compression-01 — Codes préfixes : de Shannon-Fano à Huffman

**Navigation** : [Index de la série](README.md) | [Version Python](Compression-01-ShannonFano-Prefixe-Python.ipynb) | [Série Complexity — voisine](../Complexity/README.md)

> **Public : Découverte.** Prérequis : bases de C# et .NET 9 ou ultérieur. Ce carnet constitue le socle du
> premier pli de la sous-série Compression (format Origami : la série se déploie
> pli par pli, voir l'EPIC #18706). On y construit, en C# à partir de la bibliothèque standard, la notion de code
> préfixe, l'inégalité de Kraft, puis deux algorithmes de compression historiques :
> Shannon-Fano (1949) et Huffman (1952), avant de voir pourquoi le second bat
> toujours le premier.
>
> **Durée estimée** : 45 minutes.
>
> **Ce que vous saurez faire** : décoder un code préfixe sans jamais revenir en
> arrière ; vérifier qu'une famille de longueurs est réalisable (Kraft) ; construire
> Shannon-Fano et Huffman ; mesurer l'écart à l'entropie, la borne théorique.
>
> **Au fil des sections** : longueur fixe (section 1), propriété préfixe et décodage
> en un passage (section 2), inégalité de Kraft (section 3), Shannon-Fano par coupe
> descendante (section 4), Huffman par fusion ascendante (section 5), comparaison
> sur cinq distributions (section 6), entropie comme borne absolue (section 7).
> Six exercices parcourent le carnet.

In [1]:
using Microsoft.DotNet.Interactive.Formatting;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Xml.Linq;
using System.Globalization;
using System.Security.Cryptography;
// Sorties numeriques deterministes d'une machine a l'autre : culture invariante
// (separateur decimal point, comme les valeurs citees dans la prose du carnet).
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
Console.WriteLine($"Imports OK : BCL .NET {Environment.Version}, algorithmes et SVG inline");
record PlotSvg(string Content);
Formatter.Register<PlotSvg>((v,w)=>w.Write(v.Content),"text/html");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Imports OK : BCL .NET 10.0.15, algorithmes et SVG inline


## 1. Longueur fixe, longueur variable

Un fichier est une suite de symboles. Pour stocker 8 symboles différents (disons
les notes de musique de do à do), la solution la plus simple donne à chacun un
mot binaire de même longueur : 3 bits par symbole, parce que 2^3 = 8. Chaque
symbole occupe exactement 3 bits, de `000` à `111` — ni plus, ni moins.

Cette uniformité est un gaspillage dès que les symboles ne sont pas équiprobables :
en français, le « e » est bien plus fréquent que le « k ». Sur un million de notes,
donner 3 bits au « do » qui revient 400 000 fois et 3 bits au « la » qui revient
4 000 fois, c'est gaspiller 396 000 bits. L'idée de la compression **sans perte**
est simple à énoncer : donner des mots courts aux symboles fréquents, des mots longs
aux symboles rares — et pouvoir relire le message sans aucune ambiguïté.

In [2]:
var symboles8 = new[] {"do", "re", "mi", "fa", "sol", "la", "si", "do2"};
int bitsFixes = (int)Math.Ceiling(Math.Log2(symboles8.Length));
var message = new[] {"do", "mi", "sol", "do", "la", "si", "do", "re", "mi", "do"};
Console.WriteLine($"{symboles8.Length} symboles distincts -> {bitsFixes} bits par symbole en longueur fixe\n");
Console.WriteLine("symbole  | code fixe | bits");
for (int i=0;i<symboles8.Length;i++) Console.WriteLine($"{symboles8[i],-8} | {Convert.ToString(i,2).PadLeft(3,'0'),-9} | {bitsFixes}");
Console.WriteLine($"\nmessage de {message.Length} notes -> {message.Length*bitsFixes} bits au total");

8 symboles distincts -> 3 bits par symbole en longueur fixe



symbole  | code fixe | bits


do       | 000       | 3


re       | 001       | 3


mi       | 010       | 3


fa       | 011       | 3


sol      | 100       | 3


la       | 101       | 3


si       | 110       | 3


do2      | 111       | 3



message de 10 notes -> 30 bits au total


**Lecture.** La sortie affiche les 8 symboles de `do` à `do2`, chacun sur
exactement 3 bits — la table monte de `000` à `111` sans exception. Le message
de 10 notes totalise 30 bits : 10 × 3, sans aucune économie. Si « do » apparaît 4 fois sur 10 dans le message, lui donner 2 bits, 3 bits
aux cinq autres symboles utilisés et 4 bits aux deux symboles inutilisés donne
26 bits : 4×2 + 6×3 = 26, contre 30 en fixe. Cette affectation préfixe a une
somme de Kraft égale à 1/4 + 5/8 + 2/16 = 1,0. C'est le problème que le code préfixe résout.

## 2. Codes préfixes : décoder sans revenir en arrière

Un code est **préfixe** si aucun mot du code n'est le début d'un autre mot du
code. C'est exactement la condition pour lire un message binaire de gauche à
droite, un seul passage : dès que les bits lus forment un mot du code, ce mot est
forcément le bon — aucun mot plus long ne peut commencer pareil. Le récepteur n'a
jamais besoin de revenir en arrière, ni de connaître la suite du message.

Un arbre binaire le rend visible : chaque symbole est une feuille, chaque bit lu
descend d'un niveau (0 à gauche, 1 à droite). Un décodage est une simple marche
de la racine aux feuilles. L'arbre du code ambigu, lui, a un nœud interne étiqueté
« do » — c'est précisément ce qui le rend inutilisable : le récepteur s'arrêterait
là, même si la suite du message prouve que c'est « re » qui était visé.

In [3]:
var codePrefixe = new Dictionary<string,string>{{"do","00"},{"re","01"},{"mi","10"},{"sol","11"}};
var codeAmbigu = new Dictionary<string,string>{{"do","0"},{"re","01"},{"mi","10"},{"sol","110"}};
(bool ok, (string,string)? collision) EstPrefixe(Dictionary<string,string> codes) { var mots=codes.Values.OrderBy(x=>x,StringComparer.Ordinal).ToArray(); for(int i=0;i<mots.Length;i++) for(int j=0;j<mots.Length;j++) if(i!=j && mots[j].StartsWith(mots[i],StringComparison.Ordinal)) return (false,(mots[i],mots[j])); return (true,null); }
List<string> Decoder(string bits, Dictionary<string,string> codes) { var inverse=codes.ToDictionary(x=>x.Value,x=>x.Key); var sortie=new List<string>(); var tampon=""; foreach(char b in bits){tampon+=b; if(inverse.TryGetValue(tampon,out var s)){sortie.Add(s);tampon="";}} if(tampon!="") throw new ArgumentException("Suffixe non décodé"); return sortie; }
var bits=string.Concat(new[]{"do","sol","re","mi"}.Select(s=>codePrefixe[s])); var (okCode,collision)=EstPrefixe(codePrefixe); var (okAmb,collisionAmb)=EstPrefixe(codeAmbigu);
Console.WriteLine($"message binaire : {bits}\ndécodé : [{string.Join(", ",Decoder(bits,codePrefixe))}]\ncode préfixe ? {okCode}\ncode ambigu préfixe ? {okAmb} — collision : {collisionAmb}");

message binaire : 00110110
décodé : [do, sol, re, mi]
code préfixe ? True
code ambigu préfixe ? False — collision : (0, 01)


**Lecture.** La sortie décode `00110110` en `['do', 'sol', 're', 'mi']` —
quatre symboles lus d'un seul trait, sans retour arrière. Le test confirme : le
code à 4 symboles est bien préfixe (`True`), tandis que le code ambigu échoue
(`False`) car « 0 » (do) est le début de « 01 » (re). À la lecture de `01...`,
impossible de savoir si c'est do suivi d'autre chose, ou re. La propriété préfixe
n'est pas une option de confort : c'est la condition d'existence du décodage en un
passage.

## 3. L'inégalité de Kraft : ce qu'un code préfixe peut se permettre

Soit un code de $n$ mots de longueurs $\ell_1, \ell_2, \ldots, \ell_n$. Le code
est réalisable en préfixe **si et seulement si** :

$$\sum_{i=1}^{n} 2^{-\ell_i} \;\leq\; 1 \qquad \text{(inégalité de Kraft)}$$

Intuition : dans l'arbre binaire infini, un mot de longueur $\ell$ occupe une
feuille à profondeur $\ell$, c'est-à-dire une fraction $2^{-\ell}$ des feuilles
de cette profondeur. Les feuilles choisies ne doivent pas se chevaucher : leurs
fractions totalisent au plus 1. Si la somme dépasse 1, deux mots partagent
nécessairement un ancêtre commun — le code n'est pas préfixe.

On vérifie numériquement quatre familles de longueurs, et l'on voit aussi le cas
d'égalité : quand la somme vaut exactement 1, l'arbre est **complet** — aucun bit
perdu, chaque branche mène à un symbole utile. C'est le cas idéal, atteint par
exemple par le code fixe de la section 1.

In [4]:
double SommeKraft(IEnumerable<int> longueurs) => longueurs.Sum(l=>Math.Pow(2,-l));
var exemples = new Dictionary<string,int[]>{{"code fixe 3 bits x 8",Enumerable.Repeat(3,8).ToArray()},{"préfixe 4 symboles (2 bits)",new[]{2,2,2,2}},{"arbre complet 1+2+3+3",new[]{1,2,3,3}},{"impossible : 1+1+...",new[]{1,1,2,3}}};
Console.WriteLine("exemple                       | somme Kraft | réalisable en préfixe"); foreach(var (nom,ls) in exemples) Console.WriteLine($"{nom,-29} | {SommeKraft(ls),11:F4} | {(SommeKraft(ls)<=1+1e-12?"oui":"NON")}");

exemple                       | somme Kraft | réalisable en préfixe


code fixe 3 bits x 8          |      1.0000 | oui


préfixe 4 symboles (2 bits)   |      1.0000 | oui


arbre complet 1+2+3+3         |      1.0000 | oui


impossible : 1+1+...          |      1.3750 | NON


**Lecture.** La sortie teste quatre familles de longueurs : trois satisfont
Kraft avec une somme de exactement 1,0000 — le code fixe à 3 bits, le préfixe à
2 bits, et l'arbre complet 1+2+3+3. La quatrième, « 1+1+... », atteint 1,3750 et
échoue : deux mots de longueur 1 occupent déjà les deux seules feuilles de
profondeur 1, il ne reste aucune place pour les autres symboles. Kraft ne dit pas
lequel des codes réalisables est *bon* — seulement lesquels existent.

### Exercice 1 — Kraft en pratique

Un format de fichier prétend coder 6 symboles avec les longueurs $[1, 3, 4, 4, 5, 5]$.

**Étape 1 :** écrire `SatisfaitKraft(longueurs)` qui renvoie `true` si la somme de Kraft est inférieure ou égale à 1 (à $10^{-9}$ près), `false` sinon.

**Étape 2 :** vérifier les longueurs du format ci-dessus, puis celles du code de la section 4 (`[1, 2, 3, 4, 5, 5]` — Shannon-Fano sur les 6 notes `(do, ré, mi, fa, sol, la)` ; la table affichée dans la section 4).

**Indice :** la fonction `SommeKraft` de la section 3 fait déjà la moitié du travail — il ne reste que la comparaison.


In [5]:
bool? SatisfaitKraft(IEnumerable<int> longueurs) { /* TODO etudiant : sum(2^-l) <= 1 à 1e-9 près */ return null; }
int[] longueursFormat={1,3,4,4,5,5}; if(SatisfaitKraft(longueursFormat) is bool r){if(!r||SatisfaitKraft(new[]{1,1})!=true||SatisfaitKraft(new[]{1,1,1})!=false||SatisfaitKraft(new[]{3,3,3})!=true)throw new Exception("Kraft incorrect sur fixtures variées");Console.WriteLine("Exercice 1 vérifié sur quatre fixtures");}else Console.WriteLine("Exercice à compléter : SatisfaitKraft retourne encore null");

Exercice à compléter : SatisfaitKraft retourne encore null


## 4. Shannon-Fano : couper la masse en deux (1949)

L'algorithme de Shannon-Fano part des symboles triés par probabilité décroissante,
puis divise récursivement le bloc en deux groupes de masses aussi égales que
possible ; un `0` est ajouté aux mots du premier groupe, un `1` au second.

C'est un découpage **descendant** (top-down) : on décide du premier bit des mots
les plus fréquents avant de savoir comment le reste va s'agencer. La règle est
gloutonne : à chaque étape, on coupe là où l'équilibre est le meilleur. Simple à
comprendre, simple à programmer — mais cette myopie locale peut coûter cher sur
certaines distributions, comme on le verra en section 6.

In [6]:
List<(string s,double p)> Distribution = new(){("do",.40),("re",.25),("mi",.15),("fa",.10),("sol",.06),("la",.04)};
Dictionary<string,string> ShannonFano(List<(string s,double p)> probs) { var blocs=probs.OrderByDescending(x=>x.p).ToList(); var codes=blocs.ToDictionary(x=>x.s,x=>""); void Couper(List<(string s,double p)> bloc){if(bloc.Count<=1)return;double total=bloc.Sum(x=>x.p),cum=0,best=double.PositiveInfinity;int k=0;for(int i=0;i<bloc.Count-1;i++){cum+=bloc[i].p;double e=Math.Abs(2*cum-total);if(e<best){best=e;k=i;}}for(int i=0;i<bloc.Count;i++)codes[bloc[i].s]+=(i<=k?"0":"1");Couper(bloc.Take(k+1).ToList());Couper(bloc.Skip(k+1).ToList());} Couper(blocs);return codes;}
var sf=ShannonFano(Distribution); Console.WriteLine("symbole | proba | code SF | longueur"); foreach(var (s,p) in Distribution) Console.WriteLine($"{s,-7} | {p,5:F2} | {sf[s],-7} | {sf[s].Length}");

symbole | proba | code SF | longueur


do      |  0.40 | 0       | 1


re      |  0.25 | 10      | 2


mi      |  0.15 | 110     | 3


fa      |  0.10 | 1110    | 4


sol     |  0.06 | 11110   | 5


la      |  0.04 | 11111   | 5


**Lecture.** La sortie déroule l'arbre de Shannon-Fano : `do` (40 %) hérite du
mot `0` — 1 bit ; `re` (25 %) obtient `10` — 2 bits ; puis `mi` 3 bits, `fa` 4
bits, et les deux plus rares `sol` et `la` 5 bits chacun. La masse totale vaut 1 ;
à chaque coupe, le « plus équilibré » des coupages possibles est choisi — mais ce
choix est local : il optimise la coupe présente, pas la longueur finale des mots. Le code optimal (Huffman), construit dans l'autre sens, minimise la longueur moyenne : la section 5 le confronte à ce découpage sur la même distribution, la section 6 sur des distributions où les deux ne coïncident pas.

### Exercice 2 — Encoder, décoder, vérifier

**Étape 1 :** écrire `Encoder(message, codes)` qui renvoie la chaîne binaire
obtenue en remplaçant chaque symbole par son mot de code.

**Étape 2 :** écrire `DecoderExercice(bits, codes)` qui relit cette chaîne en un seul
passage, à l'aide d'un tampon qui grossit bit à bit et se vide dès qu'il forme
un mot du code (`Decoder` de la section 2 montre le principe).

**Étape 3 :** vérifier l’aller-retour sur `["do", "sol", "re", "mi"]` avec le code
de la section 2 : l'encodage puis le décodage doivent redonner la liste exacte.

**Indice :** construire une fois le dictionnaire inverse `{mot: symbole}` avant
de boucler sur les bits.

In [7]:
#nullable enable annotations
string? Encoder(IEnumerable<string> msg,Dictionary<string,string> codes) { /* TODO etudiant : concaténer les mots de code */ return null; }
List<string>? DecoderExercice(string bits,Dictionary<string,string> codes) { /* TODO etudiant : décoder en un passage */ return null; }
var messageTest=new[]{"do","sol","re","mi"};if(Encoder(messageTest,codePrefixe) is string bits2&&DecoderExercice(bits2,codePrefixe) is List<string> dec2){var mAlt=new[]{"mi","do","do","sol"};var bAlt=Encoder(mAlt,codePrefixe);if(bits2!="00110110"||!dec2.SequenceEqual(messageTest)||bAlt is null||DecoderExercice(bAlt,codePrefixe) is not List<string> dAlt||!dAlt.SequenceEqual(mAlt))throw new Exception("Aller-retour incorrect sur messages distincts");Console.WriteLine("Exercice 2 vérifié sur deux messages distincts");}else Console.WriteLine("Exercice à compléter : Encoder/DecoderExercice retournent encore null");

Exercice à compléter : Encoder/DecoderExercice retournent encore null


## 5. Huffman : fusionner depuis les feuilles (1952)

Huffman procède à l'inverse : **ascendant** (bottom-up). On part des symboles
seuls, et l'on fusionne à chaque étape les deux nœuds de plus petite masse, le
nœud fusionné prenant leur somme. Le premier bit assigné l'est donc aux symboles
les *plus rares* — c'est le dernier bit décidé qui va aux fréquents. L'arbre se
construit de bas en haut, et la profondeur d'une feuille donne la longueur de son
mot.

L'algorithme est optimal : parmi tous les codes préfixes d'une distribution
donnée, Huffman minimise la longueur moyenne (preuve par échange, hors scope ici).
Il ne garantit pas l'unicité — plusieurs arbres optimaux peuvent exister — mais il
garantit le coût minimal.

In [8]:
Dictionary<string,string> Huffman(List<(string s,double p)> probs) { var q=new PriorityQueue<List<string>,(double,int)>(); int order=0; foreach(var (s,p) in probs) q.Enqueue(new(){s},(p,order++)); var codes=probs.ToDictionary(x=>x.s,x=>""); while(q.Count>1){q.TryDequeue(out var g,out var pg);q.TryDequeue(out var d,out var pd);foreach(var s in g!)codes[s]="0"+codes[s];foreach(var s in d!)codes[s]="1"+codes[s];q.Enqueue(g!.Concat(d!).ToList(),(pg.Item1+pd.Item1,order++));}return codes;}
var hu=Huffman(Distribution); Console.WriteLine("symbole | proba | code SF | code Huffman | longueurs"); foreach(var (s,p) in Distribution) Console.WriteLine($"{s,-7} | {p,5:F2} | {sf[s],-7} | {hu[s],-12} | {sf[s].Length} / {hu[s].Length}");

symbole | proba | code SF | code Huffman | longueurs


do      |  0.40 | 0       | 0            | 1 / 1


re      |  0.25 | 10      | 10           | 2 / 2


mi      |  0.15 | 110     | 110          | 3 / 3


fa      |  0.10 | 1110    | 1110         | 4 / 4


sol     |  0.06 | 11110   | 11111        | 5 / 5


la      |  0.04 | 11111   | 11110        | 5 / 5


**Lecture.** Sur cette distribution presque harmonieuse, les deux algorithmes
attribuent les mêmes longueurs — 1, 2, 3, 4, 5, 5 bits — mais les mots diffèrent :
`sol` passe de `11110` (Shannon-Fano) à `11111` (Huffman), et `la` de `11111` à
`11110`. Les deux codes sont valides, le coût est identique. Ce n'est pas toujours
le cas : la section suivante exhibe des distributions où la coupe descendante de
Shannon-Fano est mesurable-ment moins bonne.

## 6. Le match : quand Shannon-Fano perd

Shannon-Fano choisit chaque coupe en équilibrant les masses *locales* ; Huffman
choisit chaque fusion en prenant les *plus petites masses restantes*. Ces deux
gloutonneries ne coïncident pas toujours : sur certaines distributions, la
première coupe de Shannon-Fano est « équilibrée » mais mal placée, et l'arbre
final laisse passer quelques centièmes de bit par symbole.

La section 7 montrera que ces centièmes ne sont pas anecdotiques : ils séparent un
code optimal d'un code simplement correct, et ils se mesurent en mégaoctets sur un
vrai fichier.

In [9]:
double LongueurMoyenne(List<(string s,double p)> probs,Dictionary<string,string> codes)=>probs.Sum(x=>x.p*codes[x.s].Length);
double Entropie(List<(string s,double p)> probs)=>-probs.Sum(x=>x.p*Math.Log2(x.p));
var banque=new Dictionary<string,List<(string s,double p)>> {
["uniforme 8"]=Enumerable.Range(0,8).Select(i=>(s:$"s{i}",p:1.0/8)).ToList(),
["harmonieuse (sec. 4)"]=Distribution,
["fib-like"]=new[]{1,1,2,3,5,8,13,21}.Select((w,i)=>(s:$"x{i}",p:(double)w)).ToList(),
["discriminante"]=new(){("A",.35),("B",.17),("C",.17),("D",.16),("E",.15)},
["très skew"]=new(){("A",.39),("B",.21),("C",.14),("D",.13),("E",.13)}};
Console.WriteLine("distribution           | H      | L SF   | L Hu   | SF-Hu"); foreach(var (nom,probs) in banque){double t=probs.Sum(x=>x.p);var norm=probs.Select(x=>(x.s,x.p/t)).ToList();double ls=LongueurMoyenne(norm,ShannonFano(norm)),lh=LongueurMoyenne(norm,Huffman(norm));Console.WriteLine($"{nom,-23} | {Entropie(norm):F4} | {ls:F4} | {lh:F4} | {ls-lh:+0.0000;-0.0000;0.0000}");}

distribution           | H      | L SF   | L Hu   | SF-Hu


uniforme 8              | 3.0000 | 3.0000 | 3.0000 | 0.0000


harmonieuse (sec. 4)    | 2.2008 | 2.2500 | 2.2500 | 0.0000


fib-like                | 2.3714 | 2.4444 | 2.4444 | 0.0000


discriminante           | 2.2328 | 2.3100 | 2.3000 | +0.0100


très skew               | 2.1650 | 2.2600 | 2.2200 | +0.0400


**Lecture.** La sortie compare cinq distributions. Sur les trois premières —
uniforme, harmonieuse, fib-like — égalité parfaite entre Shannon-Fano et Huffman.
Sur les deux dernières, Shannon-Fano perd : 2,3100 contre 2,3000 bits sur «
discriminante », puis 2,2600 contre 2,2200 sur « tres skew ». Le mécanisme sur «
discriminante » : Huffman donne 1 bit au symbole à 35 % (fusion 16+15, puis 17+17,
puis 31+34, puis 35 contre le reste), tandis que Shannon-Fano, en cherchant une
coupe équilibrée 52/48 dès le premier pas, condamne ce même symbole à 2 bits.
Quelques centièmes de bit par symbole, multipliés par des milliards de symboles,
font des mégaoctets.

### Exercice 3 — Battez Shannon-Fano

**Étape 1 :** écrire `EcartSfHuffman(probs)` qui renvoie la différence
$L_{SF} - L_{Huffman}$ d'une distribution (négatif ou nul si Shannon-Fano ne
perd pas).

**Étape 2 :** la faire tourner sur la distribution « discriminante » de la
section 6 et retrouver $+0{,}01$ bit par symbole.

**Étape 3 :** chercher à la main une distribution de 5 symboles qui maximise
cet écart. Essayez de dépasser $0{,}04$ bit par symbole.

**Indice :** pour que Shannon-Fano perde, il faut que sa *première coupe
équilibrée* sépare mal les symboles fréquents — une masse dominante autour de
0,35 à 0,40 et quatre petites masses comparables font l'affaire.

In [10]:
double? EcartSfHuffman(List<(string s,double p)> probs) { /* TODO etudiant : L(SF)-L(Huffman) */ return null; }
var maDistribution=new List<(string s,double p)>{("A",.35),("B",.17),("C",.17),("D",.16),("E",.15)};if(EcartSfHuffman(maDistribution) is double e){var other=EcartSfHuffman(new(){("a",.5),("b",.25),("c",.15),("d",.1)});if(!double.IsFinite(e)||Math.Abs(e-.01)>1e-9||other is not double x||!double.IsFinite(x)||Math.Abs(x)>1e-9)throw new Exception("Écart SF incorrect");Console.WriteLine($"Exercice 3 vérifié : écart={e:F4}, seconde distribution aussi contrôlée");}else Console.WriteLine("Exercice à compléter : EcartSfHuffman retourne encore null");

Exercice à compléter : EcartSfHuffman retourne encore null


## 7. L'entropie comme borne

Peut-on faire mieux que Huffman ? La réponse tient en une inégalité : pour
**tout** code préfixe d'une source de distribution $p$, la longueur moyenne $L$
vérifie

$$H(p) \;\leq\; L, \qquad H(p) = -\sum_i p_i \log_2 p_i \quad \text{(entropie de Shannon)}$$

et Huffman reste à moins de 1 bit de cette borne : $L_{Huffman} < H(p) + 1$.
Pour une source dont les probabilités sont des puissances de 1/2, un code
préfixe symbole par symbole peut atteindre H (par exemple l’uniforme à 8 symboles).
Dans les autres cas, le codage par blocs permet d’approcher H — objet des plis
suivants de la série.

Le nuage de points ci-dessous visualise cette double contrainte : chaque
distribution aléatoire génère un point $(H, L_{Huffman})$ qui tombe entre les deux
droites. Jamais en dessous de $y = H$ — aucun code ne peut battre l'entropie.
Jamais au-dessus de $y = H + 1$ — Huffman tient sa promesse.

In [11]:
Console.WriteLine("distribution           | H      | L SF   | L Hu   | H<=L ?"); foreach(var (nom,probs) in banque){double t=probs.Sum(x=>x.p);var n=probs.Select(x=>(x.s,x.p/t)).ToList();double H=Entropie(n),ls=LongueurMoyenne(n,ShannonFano(n)),lh=LongueurMoyenne(n,Huffman(n));Console.WriteLine($"{nom,-23} | {H:F4} | {ls:F4} | {lh:F4} | {H<=ls+1e-9&&H<=lh+1e-9&&lh<H+1}");}
// Gamma(shape,1) par Marsaglia-Tsang, puis normalisation Dirichlet ; graine déterministe.
double Gamma(Random r,double a){if(a<1)return Gamma(r,a+1)*Math.Pow(r.NextDouble(),1/a);double d=a-1.0/3,c=1/Math.Sqrt(9*d);while(true){double x, v; do{x=Normal(r);v=1+c*x;}while(v<=0);v=v*v*v;double u=r.NextDouble();if(u<1-.0331*x*x*x*x||Math.Log(u)<.5*x*x+d*(1-v+Math.Log(v)))return d*v;}}
double Normal(Random r){double u1=1-r.NextDouble(),u2=1-r.NextDouble();return Math.Sqrt(-2*Math.Log(u1))*Math.Cos(2*Math.PI*u2);}
var rng=new Random(42);var pts=new List<(double h,double l)>();for(int k=0;k<60;k++){double alpha=.5+3.5*rng.NextDouble();var g=Enumerable.Range(0,6).Select(_=>Gamma(rng,alpha)).ToArray();double sum=g.Sum();var ps=g.Select((v,i)=>(s:$"s{i}",p:v/sum)).ToList();var hVal=Entropie(ps);var lVal=LongueurMoyenne(ps,Huffman(ps));if(ps.Any(z=>!double.IsFinite(z.p)||z.p<=0)||Math.Abs(ps.Sum(z=>z.p)-1)>1e-12||!double.IsFinite(hVal)||!double.IsFinite(lVal)||hVal>lVal+1e-9||lVal>=hVal+1+1e-9)throw new Exception("Tirage Dirichlet hors invariants entropiques");pts.Add((hVal,lVal));}
double xmin=0,xmax=Math.Log2(6),ymin=0,ymax=4; int W=700,HH=440,L=72,R=22,T=35,B=66; double X(double x)=>L+(x-xmin)/(xmax-xmin)*(W-L-R); double Y(double y)=>HH-B-(y-ymin)/(ymax-ymin)*(HH-T-B); XNamespace ns="http://www.w3.org/2000/svg"; var root=new XElement(ns+"svg",new XAttribute("width",W),new XAttribute("height",HH),new XAttribute("viewBox",$"0 0 {W} {HH}"),new XElement(ns+"rect",new XAttribute("width","100%"),new XAttribute("height","100%"),new XAttribute("fill","white"))); root.Add(new XElement(ns+"text",new XAttribute("x",W/2),new XAttribute("y",24),new XAttribute("text-anchor","middle"),new XAttribute("font-family","sans-serif"),new XAttribute("font-size",16),"Huffman entre H et H + 1 (60 tirages)")); for(int i=0;i<=5;i++){double x=i*.5;root.Add(new XElement(ns+"line",new XAttribute("x1",X(x).ToString("F2",CultureInfo.InvariantCulture)),new XAttribute("y1",T),new XAttribute("x2",X(x).ToString("F2",CultureInfo.InvariantCulture)),new XAttribute("y2",HH-B),new XAttribute("stroke","#ddd")));root.Add(new XElement(ns+"text",new XAttribute("x",X(x).ToString("F2",CultureInfo.InvariantCulture)),new XAttribute("y",HH-B+20),new XAttribute("text-anchor","middle"),new XAttribute("font-family","sans-serif"),new XAttribute("font-size",11),x.ToString("0.0",CultureInfo.InvariantCulture)));}for(int i=0;i<=4;i++){double y=i;root.Add(new XElement(ns+"line",new XAttribute("x1",L),new XAttribute("y1",Y(y).ToString("F2",CultureInfo.InvariantCulture)),new XAttribute("x2",W-R),new XAttribute("y2",Y(y).ToString("F2",CultureInfo.InvariantCulture)),new XAttribute("stroke","#ddd")));root.Add(new XElement(ns+"text",new XAttribute("x",L-10),new XAttribute("y",(Y(y)+4).ToString("F2",CultureInfo.InvariantCulture)),new XAttribute("text-anchor","end"),new XAttribute("font-family","sans-serif"),new XAttribute("font-size",11),y.ToString(CultureInfo.InvariantCulture)));} root.Add(new XElement(ns+"line",new XAttribute("x1",X(0)),new XAttribute("y1",Y(0)),new XAttribute("x2",X(xmax)),new XAttribute("y2",Y(xmax)),new XAttribute("stroke","black"),new XAttribute("stroke-dasharray","7 5")));root.Add(new XElement(ns+"line",new XAttribute("x1",X(0)),new XAttribute("y1",Y(1)),new XAttribute("x2",X(xmax)),new XAttribute("y2",Y(xmax+1)),new XAttribute("stroke","#c33"),new XAttribute("stroke-dasharray","2 4")));foreach(var p in pts)root.Add(new XElement(ns+"circle",new XAttribute("cx",X(p.h).ToString("F2",CultureInfo.InvariantCulture)),new XAttribute("cy",Y(p.l).ToString("F2",CultureInfo.InvariantCulture)),new XAttribute("r",3.5),new XAttribute("fill","#1f77b4")));root.Add(new XElement(ns+"text",new XAttribute("x",W/2),new XAttribute("y",HH-15),new XAttribute("text-anchor","middle"),new XAttribute("font-family","sans-serif"),"Entropie H (bits)"));root.Add(new XElement(ns+"text",new XAttribute("transform",$"translate(18 {HH/2}) rotate(-90)"),new XAttribute("text-anchor","middle"),new XAttribute("font-family","sans-serif"),"Longueur moyenne L (bits)"));root.Add(new XElement(ns+"text",new XAttribute("x",W-300),new XAttribute("y",48),new XAttribute("font-family","sans-serif"),new XAttribute("font-size",11),"● Points Huffman  — y = H  - - y = H + 1"));var svg=new XDocument(root).ToString(SaveOptions.DisableFormatting);display(new PlotSvg(svg));Console.WriteLine($"Nuage : {pts.Count} distributions ; paramètre gamma tiré uniformément sur [0,5; 4], graine 42. H min={pts.Min(p=>p.h):F3}, H max={pts.Max(p=>p.h):F3}, écart L−H max={pts.Max(p=>p.l-p.h):F3}.");

distribution           | H      | L SF   | L Hu   | H<=L ?


uniforme 8              | 3.0000 | 3.0000 | 3.0000 | True


harmonieuse (sec. 4)    | 2.2008 | 2.2500 | 2.2500 | True


fib-like                | 2.3714 | 2.4444 | 2.4444 | True


discriminante           | 2.2328 | 2.3100 | 2.3000 | True


très skew               | 2.1650 | 2.2600 | 2.2200 | True


Huffman entre H et H + 1 (60 tirages) 0.0 0.5 1.0 1.5 2.0 2.5 0 1 2 3 4 Entropie H (bits) Longueur moyenne L (bits) ● Points Huffman — y = H - - y = H + 1

Nuage : 60 distributions ; paramètre gamma tiré uniformément sur [0,5; 4], graine 42. H min=1.120, H max=2.565, écart L−H max=0.239.


**Lecture.** La sortie affiche cinq distributions, toutes avec `H ≤ L` vérifié
(`True`) : l'entropie de l'uniforme vaut 3,0000 bits pour une longueur moyenne
identique ; sur « harmonieuse », H = 2,2008 et L = 2,2500. Le nuage de 60 points
confirme : chaque point tombe strictement entre la borne noire y = H (aucun code
préfixe ne peut descendre dessous) et la rouge y = H + 1 (Huffman le garantit).
L'écart à la borne noire se resserre quand la distribution se rapproche de
puissances de 1/2.

## Exercices — récapitulatif des six objectifs

Les six exercices restent de vrais travaux à compléter, répartis au moment où les notions apparaissent. Les exemples résolus des sections précédentes restent disponibles comme modèles.

| Exercice | Travail à réaliser | Emplacement |
|---|---|---|
| 1 | Tester une famille avec Kraft | Après la section 3 |
| 2 | Encoder, décoder et vérifier un aller-retour | Après la section 2 |
| 3 | Mesurer l'écart Shannon-Fano/Huffman | Après la section 6 |
| 4 | Mesurer fréquences, coût et UTF-8 sur du texte Unicode | Ici |
| 5 | Comparer les profondeurs et atteindre 9 avec Fibonacci | Ici |
| 6 | Mesurer l'écart à l'entropie sur deux symboles | Ici |

Chaque fonction à compléter reste stubbée et chaque garde ne s'active que si une réponse est fournie.

### Exercice 4 — Un vrai texte, mesuré de bout en bout

Les distributions des sections précédentes étaient données. Un compresseur
commence par une mesure : compter les symboles d'un texte réel.

**Étape 1 :** écrire `Frequences(texte)` qui renvoie une liste
`List<(string symbole, double probabilité)>` triée par symbole, de probabilités normalisées
(somme égale à 1) — la forme attendue par `Huffman`.

**Étape 2 :** écrire `CoutBinaire(texte, codes)` qui renvoie le nombre total de
bits du texte encodé, c'est-à-dire la somme des longueurs des mots de chaque
symbole.

**Étape 3 :** sur la variable `texte`, comparer `CoutBinaire(...)` à la référence
`8 * texte.EnumerateRunes().Count()` bits (8 bits par caractère : l'encodage ASCII de référence,
valable ici sauf le `é` de « fréquent » — voir l'étape 4) et afficher le taux de compression obtenu.

**Étape 4 :** le `é` échappe à l'ASCII : en UTF-8 il occupe 2 octets. Afficher
aussi la référence `8 * Encoding.UTF8.GetByteCount(texte)` bits, puis le taux de
compression face à celle-ci. `EnumerateRunes().Count()` compte les points de code Unicode, tandis que
`Encoding.UTF8.GetByteCount(texte)` compte les octets : les deux références ne sont pas
égales — laquelle est la plus honnête ?

**Indice :** un dictionnaire de comptes (`comptes[symbole] += 1`) suffit ; la
probabilité est le compte divisé par la longueur du texte.

In [12]:
#nullable enable annotations
string texte="la compression sans perte remplace les codes de longueur fixe par des codes de longueur variable plus un symbole est fréquent plus son mot est court et plus le texte se repete plus le gain grandit";
List<(string s,double p)>? Frequences(string t) { /* TODO etudiant : compter les caractères Unicode et normaliser */ return null; }
int? CoutBinaire(string t,Dictionary<string,string> codes) { /* TODO etudiant : compter les bits du flux */ return null; }
if(Frequences(texte) is List<(string s,double p)> f&&CoutBinaire(texte,Huffman(f)) is int cost){var uni=Frequences("é🙂é");var textChars=texte.EnumerateRunes().Select(r=>r.ToString()).ToArray();var refFreq=textChars.GroupBy(x=>x).OrderBy(g=>g.Key,StringComparer.Ordinal).Select(g=>(s:g.Key,p:(double)g.Count()/textChars.Length)).ToList();var textCodes=Huffman(f);int refCost=textChars.Sum(ch=>textCodes[ch].Length);int? unicodeCost=uni is null?null:CoutBinaire("é🙂é",Huffman(uni));if(f.Count<10||!f.SequenceEqual(refFreq)||Math.Abs(f.Sum(x=>x.p)-1)>1e-9||cost!=refCost||uni is null||uni.Count!=2||uni.Any(x=>!double.IsFinite(x.p))||uni[0].s!="é"||uni[1].s!="🙂"||Math.Abs(uni[0].p-2.0/3)>1e-9||Math.Abs(uni[1].p-1.0/3)>1e-9||unicodeCost!=3||System.Text.Encoding.UTF8.GetByteCount("é🙂é")!=8)throw new Exception("Fréquences Unicode/coût invalides");Console.WriteLine($"Exercice 4 vérifié : {f.Count} caractères source, {cost} bits, fréquences Unicode séparées");}else Console.WriteLine("Exercice à compléter : Frequences/CoutBinaire retournent encore null");

Exercice à compléter : Frequences/CoutBinaire retournent encore null


### Exercice 5 — La profondeur, ou le prix de l'optimalité

Un code optimal n'est pas un code équilibré : sur une distribution très inégale,
l'arbre de Huffman dégénère en chaîne, et le symbole le plus rare descend
profondément.

**Étape 1 :** écrire `ProfondeurMax(codes)` qui renvoie la longueur du plus long
mot du code.

**Étape 2 :** afficher cette profondeur pour chaque distribution de `banque`
(section 6), puis repérer celle qui atteint le maximum pour ses symboles.

**Étape 3 :** construire une distribution de 10 symboles de poids
`[1, 1, 2, 3, 5, 8, 13, 21, 34, 55]` (suite de Fibonacci), lui appliquer
`Huffman`, et vérifier que la profondeur maximale vaut 9 — la chaîne complète —
alors que le code reste optimal.

**Indice :** `codes.Values.Max(mot => mot.Length)`. La distribution
« fib-like » de la section 6 est le même schéma à 8 symboles.

In [13]:
int? ProfondeurMax(Dictionary<string,string> codes) { /* TODO etudiant : longueur maximale */ return null; }
int[] poidsFibonacci10={1,1,2,3,5,8,13,21,34,55};var fib=poidsFibonacci10.Select((w,i)=>(s:$"f{i}",p:(double)w/143)).ToList();var c10=Huffman(fib);if(ProfondeurMax(c10) is int d){var entries=banque.Values.Select(x=>{var norm=x.Select(y=>(y.s,y.p/x.Sum(z=>z.p))).ToList();var hc=Huffman(norm);return (observed:ProfondeurMax(hc),expected:hc.Values.Max(z=>z.Length));}).ToArray();if(d!=9||entries.Length!=5||entries.Any(x=>x.observed!=x.expected)||entries.Select(x=>x.expected).Distinct().Count()<2)throw new Exception("Profondeur Huffman incorrecte");Console.WriteLine($"Exercice 5 vérifié : profondeur Fibonacci={d}; 5 distributions vérifiées");}else Console.WriteLine("Exercice à compléter : ProfondeurMax retourne encore null");

Exercice à compléter : ProfondeurMax retourne encore null


### Exercice 6 — La borne d'entropie est-elle serrée ?

La section 7 énonce `H ≤ L` et `L < H + 1`. Le second membre est une borne
supérieure : reste à savoir à quel point elle peut être atteinte.

**Étape 1 :** écrire `EcartAEntropie(probs)` qui renvoie `L_Huffman - H`.

**Étape 2 :** l'évaluer sur la distribution à deux symboles
`[("A", 0.999), ("B", 0.001)]`.

**Étape 3 :** chercher une probabilité `p` du symbole `A` telle que l'écart
dépasse 0,9 bit, puis expliquer en une phrase pourquoi le codage par blocs, et
non un meilleur code symbole par symbole, est la seule voie pour s'en approcher.

**Indice :** à deux symboles, Huffman attribue toujours 1 bit à chacun — la
longueur moyenne vaut 1 quel que soit `p`, et seule l'entropie bouge.

In [14]:
double? EcartAEntropie(List<(string s,double p)> probs) { /* TODO etudiant : L(Huffman)-H */ return null; }
var deuxSymboles=new List<(string s,double p)>{("A",.999),("B",.001)};if(EcartAEntropie(deuxSymboles) is double g){var alt=new List<(string s,double p)>{("A",.8),("B",.2)};var g2=EcartAEntropie(alt);if(!double.IsFinite(g)||Math.Abs(g-(1-Entropie(deuxSymboles)))>1e-9||g2 is not double gg||!double.IsFinite(gg)||Math.Abs(gg-(1-Entropie(alt)))>1e-9)throw new Exception("Écart entropie incorrect");Console.WriteLine($"Exercice 6 vérifié sur p=0,999 et p=0,8 : écarts {g:F4}/{g2:F4}");}else Console.WriteLine("Exercice à compléter : EcartAEntropie retourne encore null");

Exercice à compléter : EcartAEntropie retourne encore null


## À retenir

| Notion | Ce qu'elle dit | Où la voir |
|---|---|---|
| Code préfixe | aucun mot n'est préfixe d'un autre ; décodage en un passage | section 2 |
| Kraft | $\sum 2^{-\ell_i} \leq 1$ : les longueurs réalisables | section 3 |
| Shannon-Fano | coupe descendante équilibrée ; simple, parfois sous-optimal | sections 4 et 6 |
| Huffman | fusion ascendante des plus petites masses ; optimal parmi les préfixes | sections 5 et 6 |
| Entropie | $H \leq L$ pour tout code préfixe, Huffman à moins de 1 de $H$ | section 7 |

Ce premier pli a posé le socle : symboles, mots, arbres, et la borne. De la
longueur fixe uniforme (3 bits par note, 30 bits pour 10 notes) au code de Huffman
(longueur moyenne 2,25 bits sur la distribution harmonieuse), chaque section a
ajouté une contrainte : pouvoir décoder sans ambiguïté (préfixe), savoir quelles
longueurs sont réalisables (Kraft), construire le meilleur code possible (Huffman),
mesurer l'écart à la limite théorique (entropie). Les plis suivants (format
Origami, ouverts à la livraison de celui-ci — EPIC #18706) traitent du codage par
blocs qui approche la borne, des codecs réels, et du jumeau Lean des définitions.

## Vérifiez votre compréhension

**1. Pourquoi un code préfixe se décode-t-il en un seul passage, sans revenir en arrière ?**

<details>
<summary>Réponse</summary>

Parce qu'aucun mot n'est le début d'un autre : en lisant le flux bit à bit, dès
qu'on a formé un mot du code, on sait qu'aucun mot plus long ne commencera par ces
mêmes bits. La frontière entre deux mots est déterminée par la lecture, sans
hypothèse à corriger. Le code ambigu de la section 2 montre le cas inverse : les
mots `0` et `01` cohabitent, et la collision `('0', '01')` rend la frontière
indécidable — il faut revenir en arrière pour essayer l'autre découpage.

</details>

**2. Une somme de Kraft de 1,375 interdit-elle le code, ou seulement sa forme préfixe ?**

<details>
<summary>Réponse</summary>

Elle interdit la forme préfixe. C'est ce que rend la section 3 : les trois codes
réalisables (`1.0000` dans les trois cas) sont acceptés, et la ligne
`impossible : 1+1+...` à `1.3750` est rejetée. Au-delà de 1, les longueurs
demandées occupent plus que la capacité de l'arbre binaire : aucun arbre préfixe ne
peut les porter. En dessous de 1, l'inégalité est satisfaite et un arbre existe
toujours (il reste de la place perdue).

</details>

**3. Shannon-Fano et Huffman donnent la même longueur moyenne sur trois distributions, puis divergent. Qu'est-ce qui décide ?**

<details>
<summary>Réponse</summary>

La façon dont la masse se coupe. Sur `uniforme 8`, `harmonieuse` et `fib-like`, la
coupe équilibrée de Shannon-Fano tombe sur la même partition que les fusions de
Huffman : l'écart `SF-Hu` vaut `+0.0000`. Sur `discriminante` il monte à `+0.0100`,
et sur `tres skew` à `+0.0400` : quand les probabilités sont inégales, une coupe
choisie par équilibrage local n'est pas celle qu'impose la fusion des plus petites
masses, et Huffman l'emporte. L'écart mesure exactement le prix de la règle
gloutonne de Shannon-Fano.

</details>

**4. Sur la distribution `tres skew`, l'entropie vaut 2,1650 et le code de Huffman 2,2200 : le code est-il mauvais ?**

<details>
<summary>Réponse</summary>

Non. L'entropie est une borne inférieure sur les codes préfixes, pas une cible
atteignable : les longueurs sont entières. Le théorème de Huffman garantit un écart
strictement inférieur à 1 bit, et ici il vaut `0,0550` bit — très en dessous de la
garantie. Chercher à coller à l'entropie exigerait des longueurs fractionnaires,
c'est-à-dire un autre cadre (codage par blocs, plis suivants).

</details>

## Pour aller plus loin

Le pli suivant est **[Compression-02 — Huffman canonique](Compression-02-HuffmanCanonique-Python.ipynb)** :
il reprend l'arbre construit ici et le rend transmissible — mêmes longueurs,
mêmes mots, sans avoir à envoyer l'arbre.

- **Jumeau formel** : [`Compression_lean/`](Compression_lean/) porte les définitions
  de cette sous-série en Lean 4 — les propriétés énoncées ici (préfixe, Kraft) y
  deviennent des théorèmes, et l'arbre de Huffman une structure inductive.
- **Série voisine** : [Complexity](../Complexity/README.md) traite du coût de
  *calculer* ; ici on a traité du coût de *représenter*. Les deux questions se
  rejoignent dans le compromis temps de calcul contre taille de sortie.

**Références**

- C. E. Shannon, *A Mathematical Theory of Communication*, 1948.
- R. M. Fano, *The Transmission of Information*, 1949 (technique de coupe).
- D. A. Huffman, *A Method for the Construction of Minimum-Redundancy Codes*, 1952.
- T. M. Cover, J. A. Thomas, *Elements of Information Theory*, 2e éd., 2006, chap. 5.
